# **Using Gemini with Google Search Grounding in CrewAI**

In this notebook, we’ll build a **custom LLM class** that always enables
Google Search grounding for Gemini. This allows our agents to fetch
**fresh, up-to-date information from the web** instead of relying only on
the model’s training data.

---

**Why a Custom LLM?**

CrewAI provides a generic `LLM` wrapper, but it doesn’t yet expose Gemini’s
`googleSearch` grounding flag. By extending the base `LLM` class, we can
inject the right tool configuration and ensure every call to Gemini has
search enabled.

## Install dependecies

In [11]:
%uv pip install -U --quiet crewai crewai-tools

Note: you may need to restart the kernel to use updated packages.


## Set up API Key

We’ll use the **Google AI Studio API** (not Vertex).  
To run this notebook, you’ll need a Gemini API key:

🔑 **Generate your API key here:**  
[Google AI Studio – API Key Setup](https://aistudio.google.com/app/apikey)

```bash
export GEMINI_API_KEY="AIza..."  # replace with your key
```

In [12]:
import os

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print('✅ GEMINI API key set successfully!')
else:
    raise ValueError('Please enter GEMINI API key in the environment variables')

✅ GEMINI API key set successfully!


## Implementing `GeminiWithGoogleSearch`

We define a custom subclass of LLM that:

*   Forces the model to use Gemini AI Studio (e.g., gemini/gemini-2.5-pro)
*   Injects the googleSearch grounding tool
*   Passes through CrewAI’s extra arguments (from_task, etc.)
*   Enables `web_search_options` so LiteLLM actually performs the search


In [13]:
from typing import Any, List, Dict, Union
from crewai import LLM

class GeminiWithGoogleSearch(LLM):
    """
    Custom Gemini LLM using Google AI Studio (NOT Vertex) with Google Search grounding.
    - Injects the 'googleSearch' tool each call
    - Enables LiteLLM's web_search_options so search actually runs
    Requires: os.environ['GEMINI_API_KEY'] or pass api_key=...
    """

    def __init__(self, model: str | None = None, api_key: str | None = None, **kwargs):
        model = model or os.getenv("MODEL", "gemini/gemini-2.5-pro")
        api_key = GEMINI_API_KEY
        if not api_key:
            raise ValueError("Missing GEMINI_API_KEY. Set the env var or pass api_key=...")

        super().__init__(model=model, api_key=api_key, **kwargs)

    def call(
        self,
        messages: Union[str, List[Dict[str, str]]],
        tools: List[Dict] | None = None,
        callbacks: List[Any] | None = None,
        available_functions: Dict[str, Any] | None = None,
        **kwargs,
    ) -> Union[str, Any]:
        # Inject Gemini's search tool in camelCase
        _tools: List[Dict] = list(tools) if tools else []
        has_google_search = any(
            isinstance(t, dict) and ("googleSearch" in t or "google_search" in t)
            for t in _tools
        )
        if not has_google_search:
            _tools.insert(0, {"googleSearch": {}})

        return super().call(
            messages=messages,
            tools=_tools,
            callbacks=callbacks,
            available_functions=available_functions,
            **kwargs,
        )

## Test it with a single Agent

With our new custom LLM, we create a **Company Researcher** agent whose
job is to find fresh information about a given company.

We’ll test the agent by asking:

> “Fundraising information for OpenRouter Inc.”


If everything is working, Gemini should use Google Search to retrieve
recent funding announcements instead of hallucinating from old data.

In [14]:
from crewai import Agent

gemini_with_search = GeminiWithGoogleSearch()

researcher = Agent(
    role="Company Researcher",
    goal="Fetch current information for a given company and return proper Markdown",
    backstory="Specializes in searching with Google-grounded Gemini responses.",
    llm=gemini_with_search,
    verbose=True,
)

output = researcher.kickoff(messages="Fundraising information for OpenRouter Inc.")

╭─────────────────────────────────────────────── LiteAgent Started ───────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Session Started                                                                                      │
│  Name: Company Researcher                                                                                       │
│  id: 02995baa-5082-44d4-b6a7-d1f6d9174c2c                                                                       │
│  role: Company Researcher                                                                                       │
│  goal: Fetch current information for a given company and return proper Markdown                                 │
│  backstory: Specializes in searching with Google-grounded Gemini responses.                                     │
│  tools: []                                                                                                      │
│  verbose: True                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Company Researcher                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## OpenRouter Inc. Secures $40 Million in Funding Across Two Rounds                                            │
│                                                                                                                 │
│  OpenRouter Inc., a startup that provides a unified API for accessing over 400 large language models, has       │
│  raised a total of $40.5 million in funding over two rounds. This places the company's valuation at             │
│  approximately $500 million.                                                                                    │
│                                                                                                                 │
│  The company's latest funding was a **$28 million Series A round in April 2025, led by Menlo Ventures**. This   │
│  followed a **$12.5 million seed round in February 2025, led by Andreessen Horowitz**. Other notable investors  │
│  across both rounds include Sequoia Capital, Soma Capital, and angel investor Fred Ehrsam.                      │
│                                                                                                                 │
│  ### Funding Rounds at a Glance:                                                                                │
│                                                                                                                 │
│  | Date | Funding Round | Amount | Lead Investor |                                                              │
│  |---|---|---|---|                                                                                              │
│  | April 2025 | Series A | $28 million | Menlo Ventures |                                                       │
│  | February 2025 | Seed | $12.5 million | Andreessen Horowitz |                                                 │
│                                                                                                                 │
│  Founded in 2023 by Alex Atallah, co-founder of OpenSea, and Louis Vichy, OpenRouter aims to simplify the       │
│  development of AI applications by providing a single point of access to a wide array of models from providers  │
│  like OpenAI, Google, and Anthropic. Their platform is designed to help developers and enterprises seamlessly   │
│  switch between different AI models without the need to rewrite code or manage multiple contracts.              │
│                                                                                                                 │
│  The influx of capital will be used to accelerate product development, expand the number of supported AI        │
│  models, and enhance the platform's enterprise capabilities. OpenRouter has seen significant traction,          │
│  processing over 8.4 trillion tokens per month and managing an annualized inference spending that surpassed     │
│  $100 million in April 2025. This represents a tenfold increase in six months. The company monetizes its        │
│  service by taking a 5% commission on customer inference spending.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── LiteAgent Completion ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Name: Company Researcher                                                                                       │
│  id: 02995baa-5082-44d4-b6a7-d1f6d9174c2c                                                                       │
│  role: Company Researcher                                                                                       │
│  goal: Fetch current information for a given company and return proper Markdown                                 │
│  backstory: Specializes in searching with Google-grounded Gemini responses.                                     │
│  tools: []                                                                                                      │
│  verbose: True                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Raw output from the agent

In [15]:
output

LiteAgentOutput(raw="## OpenRouter Inc. Secures $40 Million in Funding Across Two Rounds\n\nOpenRouter Inc., a startup that provides a unified API for accessing over 400 large language models, has raised a total of $40.5 million in funding over two rounds. This places the company's valuation at approximately $500 million.\n\nThe company's latest funding was a **$28 million Series A round in April 2025, led by Menlo Ventures**. This followed a **$12.5 million seed round in February 2025, led by Andreessen Horowitz**. Other notable investors across both rounds include Sequoia Capital, Soma Capital, and angel investor Fred Ehrsam.\n\n### Funding Rounds at a Glance:\n\n| Date | Funding Round | Amount | Lead Investor |\n|---|---|---|---|\n| April 2025 | Series A | $28 million | Menlo Ventures |\n| February 2025 | Seed | $12.5 million | Andreessen Horowitz |\n\nFounded in 2023 by Alex Atallah, co-founder of OpenSea, and Louis Vichy, OpenRouter aims to simplify the development of AI applicati

## Rendering output nicely

Gemini often returns markdown-style text. To display it cleanly inside
a Jupyter notebook, we use `IPython.display.Markdown`.
Since Gemini’s formatting can be inconsistent, we add a small normalizer
to fix bullets, headings, and spacing before rendering.

In [17]:
from IPython.display import display, Markdown
display(Markdown(output.raw))

## OpenRouter Inc. Secures $40 Million in Funding Across Two Rounds

OpenRouter Inc., a startup that provides a unified API for accessing over 400 large language models, has raised a total of $40.5 million in funding over two rounds. This places the company's valuation at approximately $500 million.

The company's latest funding was a **$28 million Series A round in April 2025, led by Menlo Ventures**. This followed a **$12.5 million seed round in February 2025, led by Andreessen Horowitz**. Other notable investors across both rounds include Sequoia Capital, Soma Capital, and angel investor Fred Ehrsam.

### Funding Rounds at a Glance:

| Date | Funding Round | Amount | Lead Investor |
|---|---|---|---|
| April 2025 | Series A | $28 million | Menlo Ventures |
| February 2025 | Seed | $12.5 million | Andreessen Horowitz |

Founded in 2023 by Alex Atallah, co-founder of OpenSea, and Louis Vichy, OpenRouter aims to simplify the development of AI applications by providing a single point of access to a wide array of models from providers like OpenAI, Google, and Anthropic. Their platform is designed to help developers and enterprises seamlessly switch between different AI models without the need to rewrite code or manage multiple contracts.

The influx of capital will be used to accelerate product development, expand the number of supported AI models, and enhance the platform's enterprise capabilities. OpenRouter has seen significant traction, processing over 8.4 trillion tokens per month and managing an annualized inference spending that surpassed $100 million in April 2025. This represents a tenfold increase in six months. The company monetizes its service by taking a 5% commission on customer inference spending.

## Further Reading & Support

📘 CrewAI Docs – Custom LLM Implementation:
https://docs.crewai.com/how-to/custom-llm

📘 CrewAI Docs – Agents & LLMs:
https://docs.crewai.com/concepts/agents

💬 CrewAI Community Forum (Q&A, discussions):
https://community.crewai.com

📘 Gemini API Docs – Grounding with Google Search:
https://ai.google.dev/gemini-api/docs/grounding